# Simple Protein Docking

Adapted for Chem 112 by Dr. Closser from workshop by Jessica A. Nash https://pdb101.rcsb.org/train/training-events/python3
If you are interested in the full workshop data including how to download molecules and proteins directly, feel free to check it out.

## Prerequisites

This notebook assumes users are familiar with protein structure and intermolecular forces
and the following python/Jupyter skills

- defining and using variables 
- executing and interpreting code
  
## Content Objectives

After completing this exercise you should be able to

- explain the concept of ligand docking
- interpret the effect of structural modifications on docking calculations
- interpret the results of a ligand docking simulation


## Process Objectives

After completing this exercise you should be able to use python to

- use rdkit to manipulate and view small molecules
- use the RCSB Search API to find a protein structure for docking.
- prepare a protein and ligand structures for input into AutoDock Vina.


<div class="alert alert-block alert-warning"> 
<strong>Important notes</strong>
    
WU = Warm Up questions; CTQ = Critical Thinking Questions.

Remember that cells must be evaluated in order for variables to be defined as expected.

Note it is ok to ignore any warnings that are displayed from rdkit. They will not affect the results.
</div>

# Warm-up Questions

**WU 1** What is a ligand? 


**WU 2** What forces are involved when it is said that a "ligand binds to a protein"?

**WU 3** What is meant by the term "ligand docking"?


# Part 1: Ligands and Modifications


As the first step we will load an "ideal" structure for a ligand obtained from the [PDB](https://www.rcsb.org/) and subsequently manipulate it using RDKit. 

By loading our molecule from an ideal structure with 3D coordinates, we can ensure that we're already near a "good structure" for our manipulated molecules and our geometry optimization will be more likely to succeed.



### Libraries used in this section

| Library    | Description     |
| :-----------: | :------------ |
| rdkit | Cheminformatics Toolkit |
| Chem | A subset of rdkit for molecule manipulation |
| IPythonConsole | A subset of rdkit to control image quality |
| Draw | A subset of rdkit for structure drawing |
| AllChem | A subset of rdkit for optimizing 3D structures |
| os         | operating system functions - handling file paths and directories. |

After completing this notebook, if you wish to dig deeper on rdkit, consider reading [Getting Started with rdkit in Python](https://www.rdkit.org/docs/GettingStartedInPython.html).



In [ ]:
# Load libraries
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw

# Configuration for displaying in Jupyter notebooks
IPythonConsole.ipython_useSVG = True  # Use SVG for higher quality images
IPythonConsole.drawOptions.addAtomIndices = True  # Show atom indices
IPythonConsole.molSize = 400,300 # Set size of image

# Read in and display a ligand
ligand = Chem.MolFromMolFile("13U_ideal.sdf")
ligand

# Modifying a ligand

We will modify [13U: N-cyclooctylglycyl-N-(4-carbamimidoylbenzyl)-L-prolinamide](https://www.rcsb.org/ligand/13U) as loaded above to create a slightly modified ligands. In the example we will substitute nitrogen for carbon in an aromatic ring. 

Please notice the index numbers attached to each atom in the ligand image generated by the previous cell. This is possible because of an earlier command that we used to display these index numbers. 

`IPythonConsole.drawOptions.addAtomIndices = True  # Show atom indices`

We will use these index numbers to tell the Python code which atoms to modify.

In [ ]:
# load a duplicate copy of 13U to manipulate
mod_ligand_N = Chem.MolFromMolFile("13U_ideal.sdf")

# change carbon in ring to a nitrogen
mod_ligand_N.GetAtomWithIdx(23).SetAtomicNum(7)

# remove any unnecessary hydrogens
atom = mod_ligand_N.GetAtomWithIdx(23) # Select our nitrogen atom
atom.SetNumExplicitHs(0) # Set the number of explicit hydrogens to 0

mod_ligand_N

<div class="alert alert-block alert-success"> 
<strong>Excerise 1</strong>

Create another modified ligand by changing a single atom. Make sure to name it something new!
</div>

**CTQ 1** What effect to you expect the change you made to have on the ligand overall? Consider polarity, size, etc.

Now that we have our manipulated molecules, we'll optimize the structures using RDKit and save them.

In [ ]:
# Optimize molecule with added nitrogen and save
from rdkit.Chem import AllChem

# Clean up structure and then add H atoms back in explicitly
Chem.SanitizeMol(mod_ligand_N)
mod_ligand_NH = Chem.AddHs(mod_ligand_N)


# Do a constrained embedding to keep the ligand in the same position
# this allows for the hydrogens to be added in reasonable locations, but keeps
# the heavy atoms in the same position
# See https://rdkit.org/docs/source/rdkit.Chem.AllChem.html#rdkit.Chem.AllChem.ConstrainedEmbed
constrained_mol = AllChem.ConstrainedEmbed(mod_ligand_NH, mod_ligand_N, useTethers=True)
constrained_mol

In [ ]:
# Perform geometry optimization
opt_N = AllChem.MMFFOptimizeMolecule(mod_ligand_NH)
mod_ligand_NH # note the original molecule is modified.

<div class="alert alert-block alert-success"> 
<strong>Excerise 2</strong>

Optimize the structure of your other modified analog.
</div>

**CTQ 2** What are the most signficant changes after optimization? Note that MMFF is classical forcefield method that treats all the atoms as masses attached by springs.

<div class="alert alert-block alert-success"> 
<strong>Excerise 3</strong>

Save all three ligands to new files by modfiying the code below.
</div>

In [ ]:
# save to new files
import os

# make modified ligand directory
os.makedirs("files_to_dock", exist_ok=True)

# copy original molecule with all hydrogens intact
ligand_H = Chem.MolFromMolFile("13U_ideal.sdf", removeHs=False)

# save modified ligands sdf file - make sure all contain hydrogens  
# and place in a folder of ligands to dock.
Chem.MolToMolFile(ligand_H, 'files_to_dock/13U.sdf')
Chem.MolToMolFile(mod_ligand_NH, 'files_to_dock/13U_mod_N.sdf')
### Add a line here for your second ligand modification!

# Part 2: Prepare Docking simulation

In this section, we will get our structure files ready for docking.
We will need to create a special file format called pdbqt, which is used by AutoDock Vina.
The PDBQT format is similar to the PDB format, but it includes additional information such as atomic charges.

We will need to complete a few steps:

1. Isolate the protein in the structure downloaded from the pdb (strip any water molecules and ligand).
2. Add hydrogens to the protein and clean up the structure.
3. Create a PDBQT file for our protein.
4. Create PDBQT files for our ligands.

### Python Libraries use in this section

| Library         | abbreviation | Purpose |
|:-------------|:---------:|:------------|
| os           | N/A      | operating system functions - handling file paths and directories. |
| nglview      | nv       | for viewing molecular structures |
| MDAnalysis     | mda | molecular dynamics library - used for reading/writing files and selecting atoms |

### Command Line Software Used in this Notebook

We'll also be adding a few command line scripts and utilities to this notebook.
Usually, these would be executed in the terminal, but we'll run them from the Jupyter interface.
These will be used to prepare our structures for docking calculations

| Software         | Purpose |
|:-------------|:---------|
| pdb2pqr      | adding hydrogens and missing atoms to protein, adjusting for pH |
| meeko        | preparing ligands for docking |

## Visualizing the protein strucure

Before we start to really work with our molecule, let's investigate the structure.
We will use a library called MDAnalysis to first process our PDB, then visualize it with a library called NGLView.

MDAnalysis is a Python library that is used to process molecular dynamics trajectories and other 3D strucure molecular files.
The core object for MDAnalysis is a "Universe" and it corresponds to a molecular system.
We can load a PDB file into MDAnalysis, then do things like measure distances in our structure or isolate particular parts.

In [ ]:
import MDAnalysis as mda

# Load into MDA universe
u = mda.Universe("2zq2.pdb")
u

After loading our PDB, we can see that we have an MDAnalysis "universe" (or molecular system) that contains 2024 atoms. 
We can inspect this structure visually using a library called NGLView.
NGLView is a molecular visualizer made to work on the web and Jupyter notebooks.
If you're interested in learning about NGLView, you can see a [video from MolSSI's first Crash Course with the PDB](https://www.youtube.com/watch?v=6QHWhycMuXc). 

In [ ]:
import nglview as nv
view = nv.show_mdanalysis(u)
view

This view looks a bit messy, and we likely want to isolate the protein and ligand for viewing.
MDAnalysis has a human readable [selection syntax](https://docs.mdanalysis.org/stable/documentation_pages/selections.html)
that allows us to isolate parts of our structure. We will take our MDAnalysis Universe (the variable `u`) and use the `select_atoms` function.
Inside this function, we will fill in what we want to select.

We will create separate variables for the protein and ligand. We can select all protein residues in MDAnalysis using the word "protein" in the `select_atoms` function. Then, we will select our ligand using `resname 13U`. This corresponds to the residue name in the PDB we downloaded.
We can also select waters in the structure by using `"resname HOH"`.

In [ ]:
# Select protein atoms
protein = u.select_atoms("protein")
ligand = u.select_atoms("resname 13U")
water = u.select_atoms("resname HOH")

water

After selecting parts of our structure, we can add them individually to an NGLView view.
In the cell below, we visualize the protein's surface area colored by hydrophobicity (hydrophobic green, hydrophilic red).
Waters from the crystal structure are in spacefill representation, and we add the ligand in a ball and stick representation.

In [ ]:
view = nv.show_mdanalysis(protein)
view.clear_representations()
view.add_representation("surface", colorScheme="hydrophobicity")
lig_view = view.add_component(ligand)
lig_view.add_representation("ball+stick")
water_view = view.add_component(water)
water_view.add_representation("spacefill")
view

**CTQ 3** Would you classify the binding pocket? Is it hydrophilic or hydrophobic? Is this consistent with the structure of the ligand? If you cannot distinguish the colors, comment on your expectations based on the structure of the ligand.

If you rotate this structure so that you are looking at the bottom, you will be able to see our `13U` ligand bound.
Upon viewing this structure carefully, you will notice that our ligand seems to appear twice. 
If you open the PDB file to investigate, you will see the following in the ligand section:

```
HETATM 1673  C14A13U A 501      18.144  -9.216  12.088  0.61 24.22           C  
ANISOU 1673  C14A13U A 501     1755   4793   2654   1752    148   1233       C  
HETATM 1674  C14B13U A 501      18.147  -8.840  11.672  0.39 24.46           C  
ANISOU 1674  C14B13U A 501     2583   4283   2430   1765    353   1279       C  
HETATM 1675  O32A13U A 501      18.209  -8.355  11.186  0.61 24.38           O  
ANISOU 1675  O32A13U A 501     2354   5394   1514   2217    238    919       O
```

This PDB structure provides [**alternate locations**](https://proteopedia.org/wiki/index.php/Alternate_locations) for each ligand atom. 
These occur when the experimental data supports multiple positions for the same atom.
In excerpt above, you will see C14A13U and C14B13U. These are alternate locations of the same atom. 
Alternate locations can also occur in the protein with some residues.

<div class="alert alert-block alert-warning">
<strong>Selecting alternate locations using MDAnalysis</strong>  
    
By checking the [documentation page](https://docs.mdanalysis.org/stable/documentation_pages/selections.html) for MDAnalysis selections, we can see that MDAnalysis is prepared for this scenario. We will want to use the `altloc` keyword. This keyword is described as:

> altLoc alternative-location

> a selection for atoms where alternative locations are available, which is often the case with high-resolution crystal structures e.g. resid 4 and resname ALA and altLoc B selects only the atoms of ALA-4 that have an altLoc B record.

If you wanted to use MDAnalysis to select for a particular ligand location, you could use:

```python
ligand_A = u.select_atoms("resname 13U and altLoc A")
ligand_B = u.select_atoms("resname 13U and altLoc B")
```
</div>

To perform a docking calculation, we will have to isolate the protein.
This starting structure for our protein contains extra molecules like ligands and water.
You will also notice from examining our visualization that our structure does not include hydrogen atoms.
If you were to examine the PDB file, you would also see that there are some missing atoms and some of our protein residues have alternate locations marked just like thie ligand.

For docking, we will want to remove all of these extra molecules and only keep the protein.
We will also want to remove any alternate locations of residues.
We will use MDAnalysis to remove these extra molecules and save our starting protein structure as a new file.

In [ ]:
# Write protein only to new PDB file
protein.write("files_to_dock/protein_2zq2.pdb")

<div class="alert alert-block alert-danger">
<strong>Protein Charge</strong>  

After saving the protein in the cell above, you may see a warning about information for formal charges not being set in the protein. 
This warning appears because MDAnalysis did not find specific formal charge data in the PDB file and used a default value instead. 
This is not a concern for us because we will adjust the protonation states of different residues using PDB2PQR in the next steps. 
</div>

## Fixing the protein structure

Now that we've isolated our protein, we will want to ensure that we've correctly added hydrogen and fixed any missing atoms.

For fixing our protein, we will use a specialized program called PDB2PQR that is made for working with biomolecules like proteins.
The advantage of using PDB2PQR is that it will check our protein for missing atoms and multiple occupancy in the protein, and it will pick positions and add missing atoms.

<div class="alert alert-block alert-warning">
<strong>More complicated fixes: PDBFixer</strong>  

Another popular software for fixing PDB files is called [PDBFixer](https://github.com/openmm/pdbfixer). PDBFixer is an open-source tool developed by the OpenMM team and is designed to fix common problems in Protein Data Bank (PDB) files before they are used in molecular simulations. It can be used to remove unwanted molecules like water, add missing heavy atoms to incomplete residues, add hydrogen atoms where needed.

PDBFixer can be especially useful when there are missing loops or residues. In this activity, our protein is not missing any residues, but many proteins from the PDB might require more adjustment.

To see an example of preparing proteins with PDBFixer, see this [recent YouTube video](https://www.youtube.com/watch?v=pwfKE6wPaMg) posted by the Open Forcefield Initiative. In this video, the presenter first uses PDBFixer, then PDB2PQR to adjust protonation.

</div>

We will use the command-line interface of this PDB2PQR. This means that you would usually type the command below into your terminal
You can run command line commands in the Jupyter notebook by putting a `!` in front of the command. 

In [ ]:
! pdb2pqr --pdb-output=files_to_dock/protein_h.pdb --pH=7.4 files_to_dock/protein_2zq2.pdb files_to_dock/protein_2zq2.pqr --whitespace

**CTQ 4** If you look closely, you should notice the pH is set in the command above. What is the specified pH, and why is this important to specify for the simulation?

## Saving a protein PDBQT File

The PDB2PQR program outputs two files, a PDB file and a PQR file. The PDB file is similar to PDB files we have worked with before, except that it contains hydrogens.
The PQR file is another molecular file format that is similar to a PDB, but contains information about atomic radii and atomic charges.

For use with AutoDock Vina, we need our protein file to be in the "PDBQT" format. PDBQT is a specialized file format used by AutoDock Vina and other AutoDock tools. Like the PQR format, the PDBQT format can also contain partial charges. We will load our PQR file and use MDAnalysis to write a PDBQT file.

<div class="alert alert-block alert-warning">
<strong>What information do we need in the PDBQT file?</strong>

We'll be using AutoDock Vina with the "vina" scoring function (this will be explained in more detail in the next section). The vina scoring function doesn't use charges to dock, so we could have also used the PDB file without charges to convert to a PDBQT file. However, some scoring functions do use partial charges.

</div>

In [ ]:
u = mda.Universe("files_to_dock/protein_2zq2.pqr")
u.atoms.write("files_to_dock/2zq2.pdbqt")

The PDBQT file generated by MDAnalysis includes two lines at the start of the structure that AutoDock Vina doesn't accept. 
These lines start with "TITLE" and "CRYST1". To resolve this, the following cell replaces these lines with "REMARK", which is acceptable to AutoDock Vina.

You might have also just chosen to use a different software to write your PDBQT. 
[OpenBabel](https://openbabel.org/index.html) is a popular choice. However, we are using MDAnalysis here for consistency with the rest of the workshop and to limit the number of libraries we are using.

In [ ]:
# Read in the just-written PDBQT file, replace text, and write back
with open("files_to_dock/2zq2.pdbqt", 'r') as file:
    file_content = file.read()

# Replace 'TITLE' and 'CRYST1' with 'REMARK'
file_content = file_content.replace('TITLE', 'REMARK').replace('CRYST1', 'REMARK')

# Write the modified content back to the file
with open("files_to_dock/2zq2.pdbqt", 'w') as file:
    file.write(file_content)

## Ligand Preparation

When preparing small molecule PDBQT files, you could have also chosen to use MDAnalysis or other tools.
However, we are going to use a special program for small molecules and docking called [meeko](https://github.com/forlilab/Meeko).
We choose to use meeko for our ligands because it will allow us to more easily visualize our results later.
Note that when using meeko, ligands should have hydrogens added already.

We are using the command line for meeko, similar to PDB2PQR. 
You could also choose to use the Python API for this, but the command line is simpler for common tasks like converting an SDF to a PDBQT.

In the cell below, we execute a command that converts our ligands that in we prepared in the `molecule_manipulation` notebook to a PDBQT file.

<div class="alert alert-block alert-success"> 
<strong>Excerise 4</strong>

Save all three ligands to new files by modfiying the code below.
</div>

In [ ]:
# Use meeko to prepare small molecules - using meeko helps us visualize them later.
! mk_prepare_ligand.py -i files_to_dock/13U.sdf -o files_to_dock/13U.pdbqt
! mk_prepare_ligand.py -i files_to_dock/13U_mod_N.sdf -o files_to_dock/13U_mod_N.pdbqt
### add line by analogy above for your third ligand!

# Part 3: Docking with Autodock vina

In this section, we are going to dock the ligands we generated previously with [PDB entry 2zq2](https://www.rcsb.org/structure/2zq2), a trypsin structure from the cow *Bos taurus* that we retrieved and processed in the previous notebook.

Molecular docking simulates the interaction between a small molecule (ligand) and a protein, predicting how these molecules fit together at the molecular level.
It is commonly used in fields such as drug development to design molecules to bind to enzymes or proteins.

During a docking process, AutoDock Vina evaluates numerous potential orientations and conformations of the ligand within the protein’s binding pocket. 
Each pose is assigned a score based on how well it fits, using various scoring functions. 
These functions can be physics-based, empirical, or even be based on machine learning algorithms to predict binding affinity. 
For a recent review of docking scoring functions, you can see [this publication](https://link.springer.com/article/10.1007/S12539-019-00327-W).

In this notebook, we'll be performing docking with the AutoDock Vina Python API using the "vina" scoring function.


### Libraries for this section

| Library         | abbreviation | Purpose |
|:-------------|:---------:|:------------|
| os           | N/A      | operating system functions - handling file paths and directories. |
| MDAnalysis     | mda | molecular dynamics library - used for reading/writing files and selecting atoms |
| vina | vina | AutoDock Vina software for Python and Jupyter notebooks |
| prolif | plf | ProLIF (Protein-Ligand Interaction Fingerprints) - generates interaction fingerprints for complexes made of ligands, protein, DNA or RNA molecules


## Preparing for Docking: Defining a Ligand Box

When we dock our ligands to our protein, we will want to define the binding pocket. 
This is where the software will attempt to bind our ligand.
In some cases, people might use software for finding pockets in proteins to define binding sites.
However, as we saw in the previous notebook, the structure we retrieved from the PDB already had a ligand bound.
To define our binding box, we will take the position of the bound ligand from our original structure.

Luckily, MDAnalysis has tools that we can use to measure our molecule and define a binding box. 
The approach we will take in this notebook is to find the `center_of_geometry` of our ligand to define the center of our binding pocket.
Then we will consider the space around the ligand to be our box.

In [ ]:
# find the center of the ligand
import MDAnalysis as mda

original_structure = mda.Universe("2zq2.pdb")
ligand_mda = original_structure.select_atoms("resname 13U")

# Get the center of the ligand as the "pocket center"
pocket_center = ligand_mda.center_of_geometry()
print(pocket_center)

After defining the pocket center, we will define our ligand box.
One simple approach to this is to subtract the min and max of the ligand positions in each dimension.
In order to allow for ligand flexibility and potential interactions with nearby residues, we will add an additional five angstroms to each side of our box.

In [ ]:
# compute min and max coordinates of the ligand
# take the size of the ligand box to be the difference between the max and min in each direction.
lig_max = ligand_mda.positions.max(axis=0)
lig_min = ligand_mda.positions.min(axis=0)
ligand_box =  lig_max - lig_min + 5
ligand_box

The `pocket_center` and `ligand_box` variables are NumPy arrays.
However, AutoDock Vina expects them to be lists.
We convert them to lists in the cell below.

In [ ]:
pocket_center = pocket_center.tolist()
ligand_box = ligand_box.tolist()

<div class="alert alert-block alert-danger">
<strong>Error in the cell above?</strong>  

If you execute the cell above this one more than once, you will see an error occur. 
This happens because on first execution, `pocket_center` and `ligand_box` are NumPy arrays and they have methods to convert to lists.
After you've executed this once, `pocket_center` and `ligand_box` don't have a `tolist` method because they are already lists.

If you execute the cell twice and see an error, you can continue with the rest of the notebook because the variables have already been converted.

</div>

## Docking Ligands with AutoDock Vina

Now that we have PDBQT files of our protein and ligand and have defined our docking box, we are ready to perform the actual docking.
Before docking, we will make a directory to store our results.

In [ ]:
# make a directory to store our results
import os

pdb_id = "2zq2"
ligand = "13U"

os.makedirs("docking_results", exist_ok=True)

We will dock using the AutoDock Vina Python API.
First, we import `Vina` from `vina`.
We start docking with the line `v = Vina(sf_name="vina")`. 
This creates a docking calculation, `v`, and sets the scoring function to the `vina` scoring function.

In [ ]:
from vina import Vina
v = Vina(sf_name="vina",verbosity=1)

<div class="alert alert-block alert-warning">
<strong>Scoring Functions in AutoDock Vina</strong>

* Vina (`vina`): `vina` is an empirical scoring function. Binding energy is predicted as the sum of pairwise atomic interactions. It includes terms for hydrogen bonds, hydrophobic interactions, and steric clashes. The parameters for this scoring function were empirically derived from fitting data available in the PDBbind database. You can read more in the [original publication](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3041641/), or in the [Vinardo paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4865195/)
* Vinardo (`vinardo`): Vinaro stands for "Vina RaDii Optimized". It was developed to improve the scoring by adjusting atom radii and reparameterizing the empirical terms based on the PDBBIND 2013 database. You can read more in the [Vinardo paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4865195/).
* AutoDock4 (`ad4`):  uses a physics-based model and is the most computationally intensive of the available scores. The `ad4` score requires the definition of a flexible receptor, so it won't work with the PDBQT we have prepared. If you are interested in flexible docking, see [the tutorial from AutoDock Vina](https://autodock-vina.readthedocs.io/en/latest/docking_flexible.html). <strong>If you try to use the `ad4` scoring function on a receptor that was not prepared to be a flexible receptor, your notebook kernel will crash.</strong>

</div>

Then, we set the files for our ligand and receptor. We will dock just our ideal ligand first. There are two parameters to docking, the `exhaustiveness` and `n_poses`.
The exhaustiveness parameter describes the "exhaustiveness" of the docking - a higher exhaustiveness means that more ligand conformations are tried. Exhaustiveness also corresponds to the amount of computational effort used during a docking experiment. The default exhaustiveness value is 8; increasing this to 32 will give a more consistent docking result. 

In this notebook, we set the exhaustiveness to 5 to improve speed for the class. If you were to do a real docking calculation, you should consider increasing this parameter.

In [ ]:
v.set_receptor("files_to_dock/2zq2.pdbqt")
v.set_ligand_from_file("files_to_dock/13U.pdbqt")
v.compute_vina_maps(center=pocket_center, box_size=ligand_box)
v.dock(exhaustiveness=5, n_poses=5)

In [ ]:
# save the results
v.write_poses("docking_results/13U.pdbqt", n_poses=5, overwrite=True)

We can see the energies of the calculated poses by calling `energies` on the docking calculation variable.
According to the Vina documentaiton, the rows correspond to the poses, while columns correspond to different energy types.
The types of energies in the columns are `["total", "inter", "intra", "torsions", "intra best pose"]` with energy values given in kcal/mol.
The number of columns and the types of energies they represent depend on the scoring function you are using.
You can see more information in the [docs for AutoDock Vina](https://autodock-vina.readthedocs.io/en/latest/vina.html#vina.vina.Vina.energies).

In [ ]:
v.energies()


You might wish to save these energies to return to them later. 
The cell below creates a pandas dataframe and saves the energies as a comma-separated-value (CSV) file.

In [ ]:
import pandas as pd
# pandas is a library for working with tables of data.

# These are the columns for the types of energies according to AutoDock Vina docs.
column_names = ["total", "inter","intra", "torsions", "intra best pose"]

df = pd.DataFrame(v.energies(), columns=column_names)
df.head()

In [ ]:
# Save the calculated energies from docking to a CSV file
df.to_csv("docking_results/13U_energies.csv", index=False)

## Visualizing Docking Results

After performing the docking simulation and saving the energies, you might wish to visualize the poses. 
When visualizing results from molecular docking, scientists often visually inspect the 3D docked structure as well as a 2D representation called an interaction map.
We can ues a software called ProLIF (Protein-Ligand Interaction Fingerprints) to make and view these maps in the Jupyter notebook.

To generate these visualizations, we have to convert our files (again!) to the correct format.

In the step above, we wrote the poses to the file `docking_results/13U.pdbqt`. 
AutoDock Vina only writes in this file, but in order to visualize your results, we need a more standard format.
We will use meeko again to convert our poses to an SDF.
Note that meeko will only convert pdbqt files if it prepared the input docking files, which is one reason we used it in the previous notebook.

Again, we use a command line script to convert out poses.

In [ ]:
! mk_export.py docking_results/13U.pdbqt -s docking_results/13U.sdf

After converting to SDF, we can again visualize our results with ProLIF.
ProLIF requires that molecules be loaded in and has functions to load molecules in several ways.
We will use MDAnalysis for loading our proteins to ProLIF and `sdf_supplier` to load the SDFs we converted in the previous step.

In [ ]:
import prolif as plf
import MDAnalysis as mda

pdb_id = "2zq2"

protein = mda.Universe("files_to_dock/protein_h.pdb")

Next, we load our protein and ligand into ProLIF.
The function we use to do this depends on the format of our input data.

In [ ]:
protein_plf = plf.Molecule.from_mda(protein)
poses_plf = plf.sdf_supplier("docking_results/13U.sdf")

To analyze the interactions of the ligand and protein we create a molecular fingerprint object.
By default, ProLIF will calculate nine types of interactions: 'Hydrophobic', 'HBAcceptor', 'HBDonor', 'Cationic', 'Anionic', 'CationPi', 'PiCation', 'PiStacking', 'VdWContact'
However, you could set this to different interactions. You can see more information about the types of interactions in the [ProLIF docs](https://prolif.readthedocs.io/en/latest/source/modules/interaction-fingerprint.html#detecting-interactions-between-residues-prolif-interactions-interactions).

Next, we will run ProLif on our poses.
To do this calculation, we pass in our list of poses (`poses_plf`) and our ProLIF protein.

In [ ]:
# calculate fingerprint
fp = plf.Fingerprint(count=True)

# run on your poses
fp.run_from_iterable(poses_plf, protein_plf)

After running this analysis, we can visualize the interaction results.
We are using the 2D and 3D visualization maps here, but there are [many other types of analysis](https://prolif.readthedocs.io/en/latest/notebooks/docking.html#analysis) that you can perform.

In [ ]:
pose_index = 0 # corresponds to the pose with highest binding energy

fp.plot_lignetwork(poses_plf[pose_index])

# visualize ligand-protein interactions in 2D
view = fp.plot_lignetwork(poses_plf[pose_index], kind="frame", frame=pose_index, display_all=False)
view

In [ ]:
# visualize ligand-protein interactions in 3D
view = fp.plot_3d(poses_plf[pose_index], protein_plf, frame=pose_index, display_all=False) 
view

**CTQ 5** What types of intermolecular interactions are present in the pose with the highest binding affinity? 

**CTQ 6** Are these interactions the same throughout all the poses?

**CTQ 7** Do the observed interactions match the prediction from the hydrophobic interaction surface? If not, suggest why this may be the case.

<div class="alert alert-block alert-success"> 
<h3>Exercise 5</h3>

Dock at least one of the modified ligands. Note the protein does not change, but you will need to reprepare and run the code for a new ligand.

</div>

**CTQ 8** Does the modified ligand bind more or less strongly than the unmodified ligand? How do you know?

**CTQ 9** Are the interactions the same between the modified and unmodified ligand in the pose with the highest binding affinity?

## Reflection

Review the Content and Process Objectives at the beginning of the activity. Identify which of the objectives you feel you meet and those that may need more work; explain reasoning for your assessment.


Identify parts of the activity that stood out as surprising, interesting, or confusing, and explain why.


If you worked with anyone else, please give their names here.

What resources did you use for completing this activity? List them here.
